# Advanced Communications Engineering
## Tutorial 0


- Welcome to the ACE Tutorial!
- Here you will find code examples that illustrate the concepts presented in the lectures.
- This collection of Jupyter Notebooks also serve as an introduction on how to simulate the physical layer of a communications system, an important tool for a communications engineer.


### Part 1) Illustration of a Bandpass Signal

- In Part 1), we illustrate how a bandpass signal looks like in the time domain.
- More specifically, we illustrate the meaning of the amplitude and phase in a IQ diagram.
- In the Part 2), we explain more in detail other aspects of this simulation.


In [ ]:
# We start with importing the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=2)


In [ ]:
# Helper function to visualize the constellation
def visualize_constellation(constellation, title):
    plt.scatter(constellation.real, constellation.imag)
    plt.title(title)
    plt.xlabel('In-Phase')
    plt.ylabel('Quadrature')
    plt.grid()
    plt.axis('equal')
    plt.show()



In [ ]:
# Ensure reproducibility
rng = np.random.default_rng(42)  # Random number generator

# We define a custom constellation for visualization purposes
n_symbols = 4
sps = 128   # Samples per symbol
constellation = np.array([-0.5, 0.5, -1j, 1j])
visualize_constellation(constellation, f"Modified 4-QAM Constellation")

# Generate random symbol indexes and modulate
symbols_idx = # COMPLETE # Generate random symbol indexes
transmitted_symbols = constellation[symbols_idx]    # Map indexes to constellation symbols

# Now we use a rectangular pulse shaping filter
modulated_signal = np.repeat(transmitted_symbols, sps)  # Rectangular pulse shaping

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(np.arange(400), modulated_signal.real[:400], label='In-Phase')
ax[0].set_xlabel('Sample Index')
ax[0].set_ylabel('In-Phase Component')
ax[1].plot(np.arange(400), modulated_signal.imag[:400], label='Quadrature')
ax[1].set_xlabel('Sample Index')
ax[1].set_ylabel('Quadrature Component')
fig.suptitle('Modulated Signal (First 200 Samples)')
plt.show()


- Time to get the bandpass signal!
- We choose a frequency that is not too high w.r.t. the symbol rate for illustration purposes.


In [ ]:
# Here, time is normalized to the symbol period, so t = 1 => 1 symbol period
t = np.arange(len(modulated_signal)) / sps
freq = 4  # Frequency in 1 / T
carrier = # COMPLETE

# Modulate the signal to bandpass
bandpass_signal = # COMPLETE

plt.plot(t[:400], bandpass_signal[:400], label='Bandpass Signal')
plt.xlabel('Sample Index')
plt.ylabel('Amplitude')
plt.title('Bandpass Signal (First 200 Samples)')
plt.show()


##### <font color='green'>Exercise</font>
Here you can try out the following:

- Test other constellations;
- Test other values for the normalized frequency `freq`. Try increasing it to 100. What is the maximum we can set, if we want to still be able to visualize correctly?
- **(Hint: think about the sps and the Nyquist rate)**.


### Part 2) QAM Modulation and AWGNC Channel

- Our examples will focus on QAM, but you can implement other modulation formats yourself (e.g. PSK, PAM, etc.).
- At this point, we won't consider _how_ the information bits are mapped to the IQ symbols.


In [ ]:
n_symbols = 16                          # Number of symbols in the constellation
n_rows = int(np.sqrt(n_symbols))        # Number of rows in the constellation
constellation = np.array([
    # COMPLETE
    ])


In [ ]:

visualize_constellation(constellation, f'{n_symbols}-QAM Constellation') 


- Notice however that the above constellation is not normalized to have _unitary symbol energy_.
- Normalization is required so that the comparison between the different modulation formats is fair.


In [ ]:
# Here we observe one advantage of numpy arrays:
# we can perform operations on all elements of the array at once
constellation = # COMPLETE
visualize_constellation(constellation, f'{n_symbols}-QAM Constellation')    


- Now we can generate the symbols _indexes_ that will be transmitted.
- These are random integers sampled uniformly.


In [ ]:
symbols_idx = # COMPLETE # Generate random symbol indexes
transmitted_symbols = # COMPLETE    # Map indexes to constellation symbols
print(f"Symbol indexes: {symbols_idx[:6]}")    # Print first 6 symbol indexes
print(f"Transmitted symbols: {transmitted_symbols[:6]}")    # Print first 6 transmitted symbols


- During the lecture, _linear_ modulation was introduced: $$S(t) = \sum_{n=-\infty}^{\infty} A_n g(t - nT).$$

- We then have to choose a _pulse shape_ $g(t)$ to use.
- Here we chose the root-raised-cosine filter, a choice that will become clearer in the next chapters.
- The expression can be found [in this Wikipedia page](https://en.wikipedia.org/wiki/Root-raised-cosine_filter).
    - Can you implement it yourself?


In [ ]:
def get_rrc_ir(beta, span, sps):
    """Generate Root-Raised-Cosine (RRC) filter impulse response."""
    N = span * sps + 1  # Filter length (centered at 0)
    t = np.linspace(-span / 2, span / 2, N)
    # Avoid division by zero
    singularity_points = np.logical_or(t == 0, np.abs(t) == 1 / (4 * beta))
    h = np.zeros_like(t)
    h[np.logical_not(singularity_points)] = (np.sin(np.pi * t[np.logical_not(singularity_points)] * (1 - beta)) + 4 * beta * t[np.logical_not(singularity_points)] * np.cos(np.pi * t[np.logical_not(singularity_points)] * (1 + beta))) / (np.pi * t[np.logical_not(singularity_points)] * (1 - (4 * beta * t[np.logical_not(singularity_points)]) ** 2))
    # Populate with correct values at singularities
    h[t == 0] = 1.0 - beta + (4 * beta / np.pi)
    h[np.abs(t) == 1 / (4 * beta)] = (beta / np.sqrt(2)) * ((1 + 2 / np.pi) * (np.sin(np.pi / (4 * beta))) + (1 - 2 / np.pi) * (np.cos(np.pi / (4 * beta))))
    return h / np.sqrt(np.sum(h ** 2))  # Normalize filter energy


In [ ]:
# Filter parameters
beta = 0.25     # Roll-off factor
span = 10       # Filter span in symbols
sps = 8         # Samples per symbol

rrc_ir = get_rrc_ir(beta, span, sps)    
plt.plot(np.linspace(-span / 2, span / 2, len(rrc_ir)), rrc_ir)
plt.xlabel('Time (symbols)')
plt.ylabel('Amplitude')
plt.title('Root-Raised-Cosine Filter Impulse Response')
plt.xticks(np.arange(-span / 2, span / 2 + 1, 1))
plt.grid()
plt.show()


- How can we then shift and modulate this waveform? Easy: convolution!
\begin{align*}
S(t) &= \left(\sum_{n=-\infty}^{\infty} A_n \delta (t - nT)\right) * g(t) \\
&= \sum_{n=-\infty}^{\infty} A_n \delta (t - nT) * g(t) \\
&= \sum_{n=-\infty}^{\infty} A_n g(t - nT).
\end{align*}

- Therefore, we proceed with the simulated version of $$\sum_{n=-\infty}^{\infty} A_n \delta (t - nT).$$


In [ ]:
upsampled_symbols = # COMPLETE
upsampled_symbols["COMPLETE"] = # COMPLETE  # Insert symbols at every sps-th position

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].stem(np.arange(200), upsampled_symbols.real[:200], label='In-Phase')
ax[0].set_xlabel('Sample Index')
ax[0].set_ylabel('In-Phase Component')
ax[1].stem(np.arange(200), upsampled_symbols.imag[:200], label='Quadrature')
ax[1].set_xlabel('Sample Index')
ax[1].set_ylabel('Quadrature Component')
fig.suptitle('Upsampled Transmitted Symbols (First 200 Samples)')
plt.show()


In [ ]:
 # Convolve upsampled symbols with RRC filter to get the modulated signal
modulated_signal = np.convolve(upsampled_symbols, rrc_ir, mode='full')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(np.arange(200), modulated_signal.real[ :200], label='In-Phase')
ax[0].set_xlabel('Sample Index')
ax[0].set_ylabel('In-Phase Component')
ax[1].plot(np.arange(200), modulated_signal.imag[ :200], label='Quadrature')
ax[1].set_xlabel('Sample Index')
ax[1].set_ylabel('Quadrature Component')
fig.suptitle('Modulated Signal (First 200 Samples)')
plt.show()


- We add noise to the signal: $$R(t) = S(t) + Z(t)$$


In [ ]:
# Choose a SNR value
es_n0_dB = 20  # Signal-to-Noise Ratio in dB
es_n0 = # COMPLETE


- We have already fixed the average symbol energy to one, and also normalizing the energy of the pulse shape $g(t)$ guarantees that the received signal after matched filtering (correlation) will also have power equal to one.


In [ ]:
# Therefore,
n0 = 1 / es_n0  # Since symbol energy is normalized to 1

# Add noise to the modulated signal
noise = # COMPLETE
received_signal = modulated_signal + noise


- Important to notice: as discussed in the lecture, the noise is _circular_, meaning that real and imaginary parts are generated independently of each other.
- Furthermore, the noise is _white_, meaning that different time samples are _uncorrelated_.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(np.arange(200), received_signal.real[ :200], label='In-Phase with Noise')
ax[0].set_xlabel('Sample Index')
ax[0].set_ylabel('In-Phase Component')
ax[1].plot(np.arange(200), received_signal.imag[ :200], label='Quadrature with Noise')
ax[1].set_xlabel('Sample Index')
ax[1].set_ylabel('Quadrature Component')
fig.suptitle('Noisy Received Signal (First 200 Samples)')
plt.show()


- All that is left now is to implement the matched filter (MF) and then detect the symbols!


In [ ]:
demodulated_signal = np.convolve(received_signal, rrc_ir, mode='full')


In [ ]:
# We also have to compensate for delays introduced during the filtering process.
rc_ir = np.convolve(rrc_ir, rrc_ir, mode='full')
plt.plot(np.arange(len(rc_ir)), rc_ir)
plt.xlabel('Time (samples)')
plt.ylabel('Amplitude')
plt.title('Combined Raised-Cosine-Filter Impulse Response')
plt.grid()
plt.show()

plt.plot(np.arange(75, 90), rc_ir[75:90])
plt.xlabel('Time (samples)')
plt.ylabel('Amplitude')
plt.title('Combined Raised-Cosine-Filter Impulse Response (zoomed-in)')
plt.grid()
plt.show()


- By computing $g(t) * g(t)$, we observe that the peak of the impulse response has a delay of 80 samples!
- By zooming-in we can perform the sanity check that the peak of $g(t) * g(t)$ is 1, and that at the sps-th sample after the peak the response is zero, and therefore no inter-symbol interference (ISI) is observed.


In [ ]:
# Here we introduce a parameter that will be used afterwards
time_offset = 0

# Correct for delay introduced by filtering
filter_delay = (len(rrc_ir) - 1)  # Delay introduced by two RRC filters
start_idx = filter_delay + time_offset
end_idx = start_idx + len(transmitted_symbols) * sps
received_symbols = demodulated_signal[start_idx : end_idx : sps]

visualize_constellation(received_symbols, f'{n_symbols}-QAM Constellation')


In [ ]:
# We can perform a sanity check on the signal power:
received_power = np.mean(np.abs(received_symbols) ** 2)
print(f'Received Signal Power: {received_power:.4f}')   
print(f'Expected Signal Power: {1.0 + n0:.4f}')         


- Are we able to detect the symbols correctly using the MAP rule for AWGNC?
- Remember: $$\hat{m}_{\text{MAP}}[k] = \underset{m}{\text{argmin}} \| r[k] - s_m \|^2$$


In [ ]:
# Distance array:
distance_array = # COMPLETE    # Each column corresponds to a constellation point
estimated_symbols_idx = # COMPLETE

# Calculate Symbol Error Rate (SER)
n_errors = np.sum(estimated_symbols_idx != symbols_idx)
ser = n_errors / len(symbols_idx)
print(f'Symbol Error Rate (SER): {ser:.4f}')


##### <font color='green'>Exercise</font>
As follow-up work for this section, try to explore the following:

- Beginner:
    - Implement other constellations;
    - Change the value of the _roll-off_ factor `beta` in the simulation;
    - Change the $E_s/N_0$ value in the simulation;
    - The parameter _time offset_ defined earlier can be used to simulate a mismatch of the sampling time and the symbol time. How does the scatter plot look like if `time_offset` is set to something different than zero?

- Advanced:
    - Can you implement other pulse shapes?
    - Can you experiment with different noise distributions?
    - How would you modify this simulation to determine the SER for a range of $E_s/N_0$ values?
